# Inductive GATv2 for Cold-Start Bus Stop Demand Prediction
**CE902 MSc Dissertation — University of Essex**
Maria Isabel Bautista Hernandez · Supervisor: Vishal K. Singh

---
### What this notebook does
Predicts passenger boardings at **bus stops that have never been observed** (cold-start) using an inductive Graph Attention Network (GATv2), compared against 6 baselines: a naive historical-average, a classical spatial-interpolation method, and 4 feature-based tabular models.

Evaluation: **33-fold leave-borough-out cross-validation** across all London boroughs — every fold withholds an entire borough's stops, so the model must predict demand purely from environment (accessibility, land use, planned service supply) and geography, never from nearby labelled examples.

### Headline result (from the full run on the student's machine, 17 Jul 2026)
| Model | WMAPE | Notes |
|---|---|---|
| HistAvg | 1.0822 | naive borough-mean baseline |
| IDW (spatial interpolation) | 0.8487 | geography only, no features — classical cold-start baseline (Liu et al. 2017) |
| MLR (Ridge) | 0.6404 | linear direct-demand model |
| RF | 0.6428 | Random Forest |
| XGBoost | 0.6437 | gradient boosting |
| **MLP** | **0.6311** | **best model** — same features as GATv2, no graph |
| GATv2 | 0.7187 | inductive graph attention — trails MLP/RF by ~9pp |

Lower WMAPE is better. The honest finding of this project: **the graph adds
real signal over pure geography (GATv2 clearly beats IDW), but a plain
feature-based model with no graph at all (MLP/RF) still beats GATv2** under
strict spatial cross-validation. See `experiment_log.md` in the project
repository for the full analysis (33 experiments, 6 fixed bugs, per-borough
breakdown).

### Data sources (all publicly available, no data-sharing agreement required)
| Dataset | Source | Licence |
|---|---|---|
| Bus stop demand (BUSTO) | TfL Open Data — tfl.gov.uk/info-for/open-data-users/our-open-data | TfL Open Data Licence |
| Bus stop locations | TfL Open Data (Bus_Stops.csv) | TfL Open Data Licence |
| Accessibility indicators (AI23) | Verduzco Torres & McArthur (2024), UK Data Service | Open Government Licence |
| OSM POI counts | OpenStreetMap via `osmnx` | Open Database Licence (ODbL) |
| LSOA boundaries crosswalk | ONS Open Geography Portal | Open Government Licence |

### Files you need to upload
Upload these two files to your Google Drive, then update the paths in **Cell 2**:
- `stops_features_osm.csv` — 17,943 London bus stops: AI23 accessibility + OSM POI + service_coverage + coords + demand target (~3 MB)
- `route_edges.csv` — pre-extracted bus route connectivity edges, 43,220 directed edges (~500 KB)

> These files were derived from the public TfL/ONS/OSM sources above using the
> pipeline scripts `step1_aggregate_busto.py` → `step2_join_coordinates.py` →
> `step3_lsoa_features.py` → `step3b_osm_features.py` → `step3c_add_scenic.py`
> → `step3d_add_service_coverage.py` → `extract_route_edges.py` in the project
> repository. To reproduce from raw data, download from the sources above and
> run those scripts first — the raw BUSTO CSVs alone are several hundred MB,
> which is why this notebook works from the pre-aggregated feature file rather
> than raw data (keeps the Colab upload small and the notebook fast).

### Models compared
| Model | Description |
|---|---|
| HistAvg | Borough-level mean demand (naive baseline) |
| IDW | Inverse-distance-weighted spatial interpolation, k=20 nearest training stops, lat/lon only — no features (Liu et al. 2017-style cold-start baseline) |
| MLR | Ridge regression — linear direct-demand model |
| RF | Random Forest — tabular features, no graph |
| XGBoost | Gradient boosting — tabular features, no graph |
| MLP | 3-layer neural net, same capacity as GATv2 — tabular features, no graph (ablation) |
| **GATv2** | **Inductive Graph Attention Network v2 — multigraph (KNN + route edges), residual skip connection** |

**Runtime:** full 33-fold run ≈ 35-45 min on Colab T4 GPU, ≈ 2h on CPU.
Set `QUICK_RUN = True` for a 5-borough demo (a few minutes).


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

# torch-geometric and xgboost are not preinstalled in Colab; torch/sklearn are.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'scikit-learn', 'xgboost'], check=True)
print('Dependencies ready.')


In [ ]:
# ── Cell 2: Mount Drive, set file paths, choose configuration ────────────────
from google.colab import drive
drive.mount('/content/drive')

# EDIT THESE PATHS to wherever you uploaded the files in your Drive
STOPS_FEATURES_PATH = '/content/drive/MyDrive/dissertation/stops_features_osm.csv'
ROUTE_EDGES_PATH    = '/content/drive/MyDrive/dissertation/route_edges.csv'

# ── Feature-set configuration (mirrors step4_model.py's CLI flags) ───────────
OSM_ONLY  = False   # True: OSM POI + lat/lon only, no AI23
AI23_ONLY = False   # True: AI23 + lat/lon only, no OSM (ignored if OSM_ONLY)
WITH_SC   = True    # True: add service_coverage (headline config uses this)

# Set to True for a 5-borough quick demo, False for the full 33-fold run
QUICK_RUN = False


In [ ]:
# ── Cell 3: Imports and config ───────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import subgraph
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree
from xgboost import XGBRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Hyperparameters (matching the current step4_model.py, all bugs fixed) ────
K_NEIGHBORS = 5
HIDDEN_DIM  = 64
HEADS       = 4
DROPOUT     = 0.15
LR          = 5e-4
EPOCHS      = 500     # safe ceiling — early stopping triggers well before this
PATIENCE    = 20       # checks, evaluated every VAL_EVERY epochs
VAL_EVERY   = 5
VAL_FRAC    = 0.1
RF_TREES    = 150
SEED        = 42
torch.manual_seed(SEED); np.random.seed(SEED)

FEAT_COLS = [
    'employment_all_30min', 'hospitals_30min', 'gp_30min',
    'supermarkets_30min', 'pharmacies_30min',
    'primary_schools_30min', 'secondary_schools_30min', 'main_bua_30min'
]
OSM_COLS   = ['poi_residential', 'poi_shopping', 'poi_company',
              'poi_education', 'poi_entertainment', 'poi_scenic']
SC_COL     = 'service_coverage'
COORD_COLS = ['lat', 'lon']
BOROUGH_COL = 'lad_name'
TARGET_COL  = 'total_boardings'

ALL_FEAT_COLS = None  # set by prep_features() once the data is loaded


In [ ]:
# ── Cell 4: Helper functions ─────────────────────────────────────────────────

def wmape(y_true, y_pred):
    """Weighted MAPE — robust to zero-demand stops."""
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8))


def prep_features(df):
    """Builds the raw (unscaled) feature matrix per the OSM_ONLY/AI23_ONLY/WITH_SC
    flags above. All non-coordinate features are log1p-transformed (heavy right
    tail). Sets the global ALL_FEAT_COLS to the active column list."""
    global ALL_FEAT_COLS
    osm_present = [c for c in OSM_COLS if c in df.columns]
    sc_present  = [SC_COL] if (WITH_SC and SC_COL in df.columns) else []

    if OSM_ONLY and osm_present:
        active_feat = osm_present + sc_present
    elif AI23_ONLY:
        active_feat = FEAT_COLS + sc_present
    else:
        active_feat = FEAT_COLS + osm_present + sc_present

    ALL_FEAT_COLS = active_feat + COORD_COLS
    X = np.zeros((len(df), len(ALL_FEAT_COLS)), dtype=np.float32)
    for i, c in enumerate(active_feat):
        X[:, i] = np.log1p(df[c].values)
    X[:, len(active_feat)]     = df['lat'].values
    X[:, len(active_feat) + 1] = df['lon'].values
    return X


def idw_predict(train_lat, train_lon, train_y_log, test_lat, test_lon, k=20, power=2):
    """Inverse Distance Weighting — classical geostatistical cold-start baseline
    (Liu et al. 2017, NYC Citi Bike: gravity models + spatial interpolation).
    Uses ONLY lat/lon, no AI23/OSM/SC features at all."""
    train_coords = np.radians(np.column_stack([train_lat, train_lon]))
    test_coords  = np.radians(np.column_stack([test_lat, test_lon]))
    k_use = min(k, len(train_lat))
    dist, idx = BallTree(train_coords, metric='haversine').query(test_coords, k=k_use)
    dist_km = dist * 6371.0088
    w = 1.0 / np.power(dist_km + 1e-6, power)
    neighbor_y = train_y_log[idx]
    return np.sum(w * neighbor_y, axis=1) / np.sum(w, axis=1)


def build_knn_edge_index(lats, lons, k=K_NEIGHBORS):
    """K-nearest-neighbour geographic edges using Haversine distance."""
    coords_rad = np.radians(np.column_stack([lats, lons]))
    tree  = BallTree(coords_rad, metric='haversine')
    _, idx = tree.query(coords_rad, k=k + 1)
    src, dst = [], []
    for i, neighbours in enumerate(idx):
        for j in neighbours[1:]:
            src += [i, j]
            dst += [j, i]
    ei = torch.tensor([src, dst], dtype=torch.long)
    return torch.unique(ei, dim=1)


def build_route_edges_from_csv(route_edges_path):
    """Load pre-extracted route connectivity edges (consecutive stops on the
    same bus route/direction — see extract_route_edges.py)."""
    df = pd.read_csv(route_edges_path)
    src = torch.tensor(df['src'].values, dtype=torch.long)
    dst = torch.tensor(df['dst'].values, dtype=torch.long)
    ei  = torch.stack([src, dst], dim=0)
    return torch.unique(ei, dim=1)


print('Helper functions defined.')


In [ ]:
# ── Cell 5: Model definitions ────────────────────────────────────────────────

class GATv2Model(nn.Module):
    """Inductive GATv2 with a residual skip connection — critical for
    cold-start, since it lets the model weight graph aggregation vs the
    direct feature path per-fold, depending on how informative (or how
    contaminated by cross-borough noise) the neighbourhood is."""
    def __init__(self, in_dim, hidden_dim=HIDDEN_DIM, heads=HEADS, dropout=DROPOUT):
        super().__init__()
        self.c1   = GATv2Conv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        self.c2   = GATv2Conv(hidden_dim * heads, 1, heads=1, dropout=dropout, concat=False)
        self.skip = nn.Linear(in_dim, hidden_dim * heads, bias=False)

    def forward(self, x, edge_index):
        h = F.elu(self.c1(x, edge_index)) + self.skip(x)
        h = F.dropout(h, p=DROPOUT, training=self.training)
        return self.c2(h, edge_index).squeeze(-1)


class MLPModel(nn.Module):
    """Ablation: same capacity as GATv2, zero message passing."""
    def __init__(self, in_dim, hidden_dim=HIDDEN_DIM, heads=HEADS, dropout=DROPOUT):
        super().__init__()
        h = hidden_dim * heads
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(h, h // 2), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(h // 2, 1),
        )

    def forward(self, x, edge_index=None):
        return self.net(x).squeeze(-1)


print('Model classes defined.')


In [ ]:
# ── Cell 6: NN training loop + per-fold evaluation ───────────────────────────

def train_nn(model, x, y, ei, tr_pos, val_pos):
    """Early stopping on Huber loss, checked every VAL_EVERY epochs (matches
    step4_model.py). Huber is more robust than MSE to the residual heavy tail
    left after log1p(boardings)."""
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    best_val, best_w, no_imp = float('inf'), None, 0
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad()
        out  = model(x, ei)
        loss = F.huber_loss(out[tr_pos], y[tr_pos], delta=0.5)
        loss.backward(); opt.step()
        if (epoch + 1) % VAL_EVERY != 0:
            continue
        model.eval()
        with torch.no_grad():
            vl = F.huber_loss(model(x, ei)[val_pos], y[val_pos], delta=0.5).item()
        if vl < best_val - 1e-6:
            best_val, best_w, no_imp = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break
    if best_w:
        model.load_state_dict(best_w)
    return model


def run_fold(borough, df, X_raw, y_orig, full_ei, fi, nf):
    t0 = time.time()
    test_mask  = (df[BOROUGH_COL] == borough).values
    train_mask = ~test_mask
    tr_idx = np.where(train_mask)[0]
    te_idx = np.where(test_mask)[0]
    y_tr, y_te = y_orig[tr_idx], y_orig[te_idx]

    # Per-fold scaler fitted on TRAINING stops only — no leakage.
    scaler  = StandardScaler()
    X_sc_tr = scaler.fit_transform(X_raw[tr_idx])
    X_sc_te = scaler.transform(X_raw[te_idx])

    # 1. HistAvg
    ha_pred = np.full(len(te_idx), y_tr.mean())

    # 2. IDW spatial interpolation (lat/lon only, no features)
    idw_pred = np.expm1(idw_predict(df['lat'].values[tr_idx], df['lon'].values[tr_idx],
                                     np.log1p(y_tr),
                                     df['lat'].values[te_idx], df['lon'].values[te_idx]))

    # 3. MLR (Ridge)
    mlr = Ridge(alpha=1.0, random_state=SEED)
    mlr.fit(X_sc_tr, np.log1p(y_tr))
    mlr_pred = np.expm1(mlr.predict(X_sc_te))

    # 4. Random Forest
    rf = RandomForestRegressor(n_estimators=RF_TREES, max_features='sqrt',
                               min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf.fit(X_sc_tr, np.log1p(y_tr))
    rf_pred = np.expm1(rf.predict(X_sc_te))

    # 5. XGBoost
    xgbr = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                        random_state=SEED, n_jobs=-1, verbosity=0)
    xgbr.fit(X_sc_tr, np.log1p(y_tr))
    xgb_pred = np.expm1(xgbr.predict(X_sc_te))

    # Stratified val split (5 demand quantiles) so early stopping sees the
    # full boarding-demand range, not just the majority of low-demand stops.
    rng   = np.random.RandomState(SEED + fi)
    n_val = max(5, int(len(tr_idx) * VAL_FRAC))
    log_y_tr = np.log1p(y_tr)
    quantile_labels = pd.qcut(log_y_tr, q=5, labels=False, duplicates='drop')
    vp_list, n_per_q = [], max(1, n_val // 5)
    for q in range(5):
        q_idx = np.where(quantile_labels == q)[0]
        if len(q_idx) > 0:
            vp_list.extend(rng.choice(q_idx, min(n_per_q, len(q_idx)), replace=False).tolist())
    vp = np.array(vp_list)
    tp = np.setdiff1d(np.arange(len(tr_idx)), vp)
    tp_t, vp_t = torch.tensor(tp, dtype=torch.long).to(DEVICE), torch.tensor(vp, dtype=torch.long).to(DEVICE)

    all_idx  = np.concatenate([tr_idx, te_idx])
    X_sc_all = np.vstack([X_sc_tr, X_sc_te])

    x_tr  = torch.tensor(X_sc_tr, dtype=torch.float).to(DEVICE)
    x_ctx = torch.tensor(X_sc_all, dtype=torch.float).to(DEVICE)
    y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float).to(DEVICE)
    n_tr   = len(tr_idx)

    tr_t    = torch.tensor(tr_idx, dtype=torch.long)
    ei_tr,_ = subgraph(tr_t, full_ei, relabel_nodes=True, num_nodes=len(df))
    ei_tr   = ei_tr.to(DEVICE)

    all_t    = torch.tensor(all_idx, dtype=torch.long)
    ei_ctx,_ = subgraph(all_t, full_ei, relabel_nodes=True, num_nodes=len(df))
    ei_ctx   = ei_ctx.to(DEVICE)

    n_in = len(ALL_FEAT_COLS)

    # 6. MLP ablation (same capacity as GATv2, no graph)
    mlp = train_nn(MLPModel(n_in).to(DEVICE), x_tr, y_tr_t, ei_tr, tp_t, vp_t)
    mlp.eval()
    with torch.no_grad():
        mlp_pred = np.expm1(mlp(x_ctx)[n_tr:].cpu().numpy())

    # 7. GATv2 (inductive: trained on train-subgraph, test stops aggregate
    #    from training neighbours at inference time, labels never seen)
    log_cap = np.log1p(y_tr.max() * 2)
    gat = train_nn(GATv2Model(n_in).to(DEVICE), x_tr, y_tr_t, ei_tr, tp_t, vp_t)
    gat.eval()
    with torch.no_grad():
        raw = gat(x_ctx, ei_ctx)[n_tr:].cpu().numpy()
    gat_pred = np.expm1(np.clip(raw, 0, log_cap))

    result = {
        'borough': borough, 'n_test': len(te_idx),
        'HA_WMAPE':  wmape(y_te, np.clip(ha_pred, 0, None)),
        'IDW_WMAPE': wmape(y_te, np.clip(idw_pred, 0, None)),
        'MLR_WMAPE': wmape(y_te, np.clip(mlr_pred, 0, None)),
        'RF_WMAPE':  wmape(y_te, np.clip(rf_pred, 0, None)),
        'XGB_WMAPE': wmape(y_te, np.clip(xgb_pred, 0, None)),
        'MLP_WMAPE': wmape(y_te, np.clip(mlp_pred, 0, None)),
        'GAT_WMAPE': wmape(y_te, np.clip(gat_pred, 0, None)),
    }
    elapsed = int(time.time() - t0)
    print(f"  [{fi+1:2d}/{nf}] {borough:<30s} n={len(te_idx):4d}  "
          f"HA={result['HA_WMAPE']:.3f}  IDW={result['IDW_WMAPE']:.3f}  "
          f"MLR={result['MLR_WMAPE']:.3f}  RF={result['RF_WMAPE']:.3f}  "
          f"XGB={result['XGB_WMAPE']:.3f}  MLP={result['MLP_WMAPE']:.3f}  "
          f"GATv2={result['GAT_WMAPE']:.3f}  ({elapsed}s)")
    return result


print('run_fold() defined.')


In [ ]:
# ── Cell 7: Load data and build graph ────────────────────────────────────────
print('Loading stops_features_osm.csv...')
df = pd.read_csv(STOPS_FEATURES_PATH, dtype={'STOPCODE': 'string'})
df = df.dropna(subset=FEAT_COLS + ['lat', 'lon', 'total_boardings', 'lad_name'])
df = df.reset_index(drop=True)
print(f'  {len(df):,} stops, {df["lad_name"].nunique()} boroughs')

X_raw  = prep_features(df)
y_orig = df['total_boardings'].values.astype(np.float32)
print(f'Active features ({len(ALL_FEAT_COLS)}): {ALL_FEAT_COLS}')

print('Building multigraph...')
t0 = time.time()
knn_ei   = build_knn_edge_index(df['lat'].values, df['lon'].values)
print(f'  KNN edges (K={K_NEIGHBORS}): {knn_ei.shape[1]:,}  ({time.time()-t0:.1f}s)')

t0 = time.time()
route_ei = build_route_edges_from_csv(ROUTE_EDGES_PATH)
print(f'  Route edges:                {route_ei.shape[1]:,}  ({time.time()-t0:.1f}s)')

full_ei = torch.unique(torch.cat([knn_ei, route_ei], dim=1), dim=1)
print(f'  Combined (deduplicated):    {full_ei.shape[1]:,} total directed edges')


In [ ]:
# ── Cell 8: Run CV ───────────────────────────────────────────────────────────
boroughs = sorted(df[BOROUGH_COL].unique())
if QUICK_RUN:
    boroughs = boroughs[:5]
    print(f'QUICK RUN: {len(boroughs)} boroughs only')
else:
    print(f'Full run: {len(boroughs)} boroughs')

# Pre-warm PyG JIT kernel
print('Pre-warming PyG kernel...')
_n_in = len(ALL_FEAT_COLS)
_dummy_x  = torch.randn(50, _n_in).to(DEVICE)
_dummy_ei = torch.randint(0, 50, (2, 200)).to(DEVICE)
_dummy_m  = GATv2Model(_n_in).to(DEVICE)
with torch.no_grad():
    _dummy_m(_dummy_x, _dummy_ei)
del _dummy_x, _dummy_ei, _dummy_m
print('  Done.\n')

all_results = []
t_total = time.time()
for fi, borough in enumerate(boroughs):
    res = run_fold(borough, df, X_raw, y_orig, full_ei, fi, len(boroughs))
    all_results.append(res)

print(f'\nTotal time: {(time.time()-t_total)/60:.1f} min')


In [ ]:
# ── Cell 9: Results summary ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

res_df = pd.DataFrame(all_results)

models = ['HA', 'IDW', 'MLR', 'RF', 'XGB', 'MLP', 'GAT']
labels = ['HistAvg', 'IDW', 'MLR', 'RF', 'XGBoost', 'MLP', 'GATv2']
print('='*70)
print(f'RESULTS  {len(res_df)}-fold leave-borough-out  (mean +/- std | median)')
print('='*70)
for m, l in zip(models, labels):
    col = f'{m}_WMAPE'
    print(f'  {l:<8}  WMAPE={res_df[col].mean():.4f}(+/-{res_df[col].std():.4f})  '
          f'med={res_df[col].median():.4f}')
print('='*70)

# Bar chart — per-borough WMAPE
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(res_df))
w = 0.11
colors = ['#9e9e9e', '#8c564b', '#9467bd', '#4C72B0', '#2ca02c', '#DD8452', '#c0392b']
for i, (m, l, c) in enumerate(zip(models, labels, colors)):
    ax.bar(x + i*w, res_df[f'{m}_WMAPE'], w, label=l, color=c, alpha=0.9)
ax.set_xticks(x + w*3)
ax.set_xticklabels(res_df['borough'], rotation=45, ha='right', fontsize=8)
ax.axhline(1.0, color='red', linestyle='--', linewidth=0.8, label='WMAPE=1')
ax.set_ylabel('WMAPE (lower is better)')
ax.set_title('Leave-Borough-Out CV: WMAPE per borough, 7 models')
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()
plt.savefig('wmape_per_borough.png', dpi=150)
plt.show()
print('Chart saved: wmape_per_borough.png')

# Save results
res_df.to_csv('results_cv_colab.csv', index=False)
summary = pd.DataFrame([{
    'model': l,
    'WMAPE_mean': res_df[f'{m}_WMAPE'].mean(),
    'WMAPE_std':  res_df[f'{m}_WMAPE'].std(),
    'WMAPE_median': res_df[f'{m}_WMAPE'].median()
} for m, l in zip(models, labels)])
summary.to_csv('results_summary_colab.csv', index=False)
print('Results saved: results_cv_colab.csv, results_summary_colab.csv')
summary.round(4)
